# W3Q3 — Timing of Communications
**Question:** When relative to signup are emails sent? Does earlier or later communication correlate with better engagement and conversion?

**Audience:** Marketing Team  
**Data source:** `ANALYTICS.MARTS.MART_EMAIL_ENGAGEMENT` + `MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W3Q3_timing_of_communications.sql`

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W3Q3_timing_of_communications.sql') as f:
    sql = f.read()

df = query(sql)
df['first_sent_date'] = pd.to_datetime(df['first_sent_date'])
df['signup_date_clean'] = pd.to_datetime(df['signup_date_clean'])
print(f"Rows: {len(df):,}  |  Campaigns: {df['email_campaign'].nunique()}")
df.head(3)

## Send Timing by Campaign

In [ ]:
timing = (
    df.groupby('email_campaign')
      .agg(
          users=('user_id', 'nunique'),
          median_days_to_send=('days_signup_to_first_email', 'median'),
          p25=('days_signup_to_first_email', lambda x: x.quantile(0.25)),
          p75=('days_signup_to_first_email', lambda x: x.quantile(0.75)),
      )
      .sort_values('median_days_to_send')
      .reset_index()
)
display(timing)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(timing['email_campaign'], timing['median_days_to_send'], color=PALETTE[0])
ax.set_title('Median Days from Signup to First Email — by Campaign', fontsize=13, pad=12)
ax.set_xlabel('Median Days After Signup')
ax.set_ylabel('')
sns.despine()
plt.tight_layout()
plt.savefig('../output/W3Q3_send_timing_by_campaign.png', bbox_inches='tight')
plt.show()

## Engagement Rates by Campaign

In [ ]:
engagement = (
    df.groupby('email_campaign')
      .agg(
          delivery_rate=('delivery_rate', 'mean'),
          open_rate=('open_rate', 'mean'),
          click_to_open_rate=('click_to_open_rate', 'mean'),
          bounce_rate=('bounce_rate', 'mean'),
          spam_rate=('spam_rate', 'mean'),
      )
      .round(2)
      .reset_index()
)
display(engagement)

rate_cols = ['delivery_rate', 'open_rate', 'click_to_open_rate', 'bounce_rate', 'spam_rate']
engagement_plot = engagement.melt(id_vars='email_campaign', value_vars=rate_cols,
                                   var_name='Metric', value_name='Rate (%)')

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=engagement_plot, x='email_campaign', y='Rate (%)', hue='Metric', palette='muted', ax=ax)
ax.set_title('Email Engagement Rates by Campaign', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.legend(title='Metric', bbox_to_anchor=(1, 1))
sns.despine()
plt.tight_layout()
plt.savefig('../output/W3Q3_engagement_rates_by_campaign.png', bbox_inches='tight')
plt.show()

## Send Timing vs Conversion Outcome

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
outcomes = [('has_trial', 'Has Trial'), ('has_3m_subscription', 'Has 3m Sub'), ('has_12m_subscription', 'Has 12m Sub')]

for ax, (col, label) in zip(axes, outcomes):
    for val, name, color in [(True, 'Yes', PALETTE[0]), (False, 'No', PALETTE[1])]:
        data = df[df[col] == val]['days_signup_to_first_email'].dropna()
        ax.hist(data, bins=30, alpha=0.6, label=name, color=color, edgecolor='white', density=True)
    ax.set_title(f'{label}', fontsize=11)
    ax.set_xlabel('Days Signup → First Email')
    ax.set_ylabel('Density')
    ax.legend(title=label, fontsize=8)
    sns.despine(ax=ax)

fig.suptitle('Email Send Timing vs Conversion Outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/W3Q3_timing_vs_conversion.png', bbox_inches='tight')
plt.show()

## Findings

- **Email campaigns identified:** ...
- **Send timing — earliest campaign:** ...
- **Send timing — latest campaign:** ...
- **Engagement rates by campaign:** ...
- **Send timing vs conversion outcome:** ...
- **Recommendation:** ...